# Ethiopia Climate EDA (Task 2)

This notebook follows Week 0 Task 2 instructions for one country-specific analysis.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)


In [ ]:
country = 'ethiopia'
df = pd.read_csv(f'data/{country}.csv')
df['Country'] = country.title()
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j', errors='coerce')
df['Month'] = df['Date'].dt.month
df.head()


In [ ]:
df = df.replace(-999, np.nan)
duplicate_count = df.duplicated().sum()
print(f'Duplicate rows found: {duplicate_count}')
print('Duplicates are evaluated across all columns.')
if duplicate_count:
    df = df.drop_duplicates().copy()
df.describe(include=[np.number]).T


### Brief interpretation of summary statistics
- Temperature fields show baseline thermal behavior and spread.
- Precipitation summary indicates seasonality and variability.
- Wind/humidity fields capture additional atmospheric variability.

In [ ]:
missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
missing_report


In [ ]:
high_null_cols = missing_report[missing_report['missing_pct'] > 5]
print('Columns with >5% missing values:')
display(high_null_cols if not high_null_cols.empty else pd.DataFrame({'message':['None']}))


### Missing-value note
Columns above 5% missing may reduce confidence in trend/correlation outputs and should be interpreted cautiously.

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_values = pd.DataFrame({col: zscore(df[col], nan_policy='omit') for col in z_cols})
outlier_mask = (z_values.abs() > 3).any(axis=1)
print(f"Rows flagged as outliers (|Z| > 3): {int(outlier_mask.sum())}")


### Outlier handling decision
Outlier rows are retained because climate extremes are meaningful and should remain visible in analysis outputs.

In [ ]:
row_missing_ratio = df.isna().mean(axis=1)
before_rows = len(df)
df = df.loc[row_missing_ratio <= 0.30].copy()
print(f'Dropped rows with >30% missing: {before_rows - len(df)}')
weather_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_vars] = df[weather_vars].ffill()
display(df[weather_vars].isna().sum())
output_path = f'data/{country}_clean.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset exported to: {output_path}')


### Cleaning decision note
Rows with severe missingness are dropped; remaining weather gaps are forward-filled to preserve temporal continuity.

In [ ]:
monthly = (df.set_index('Date').resample('M').agg({'T2M':'mean','PRECTOTCORR':'sum'}).reset_index())
warm_idx = monthly['T2M'].idxmax()
cool_idx = monthly['T2M'].idxmin()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly['Date'], monthly['T2M'], color='tab:red')
ax.set_title(f'Monthly Average T2M ({country.title()}, 2015-2026)')
ax.set_ylabel('T2M (C)')
ax.annotate('Warmest month', (monthly.loc[warm_idx,'Date'], monthly.loc[warm_idx,'T2M']), xytext=(10,10), textcoords='offset points')
ax.annotate('Coolest month', (monthly.loc[cool_idx,'Date'], monthly.loc[cool_idx,'T2M']), xytext=(10,-15), textcoords='offset points')
plt.show()
peak_rain = monthly.nlargest(3, 'PRECTOTCORR')
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly['Date'], monthly['PRECTOTCORR'], color='tab:blue')
ax.set_title(f'Monthly Total PRECTOTCORR ({country.title()}, 2015-2026)')
ax.set_ylabel('Precipitation (mm/month)')
for _, row in peak_rain.iterrows():
    ax.annotate('Peak rainy', (row['Date'], row['PRECTOTCORR']), xytext=(0,8), textcoords='offset points', ha='center', fontsize=8)
plt.show()


### Time-series observation
Write a short interpretation of visible trends, anomalies, and rainfall seasonality.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x='T2M', y='RH2M', alpha=0.5, ax=axes[0])
axes[0].set_title('T2M vs RH2M')
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', alpha=0.5, ax=axes[1])
axes[1].set_title('T2M_RANGE vs WS2M')
plt.tight_layout()
plt.show()
top3 = corr.where(~np.eye(corr.shape[0], dtype=bool)).stack().sort_values(key=lambda x: x.abs(), ascending=False).head(3)
top3


### Top-3 correlation interpretation
Interpret the three strongest correlations shown above in a climate context.

In [ ]:
precip_skew = df['PRECTOTCORR'].skew()
plt.figure(figsize=(8, 4))
if precip_skew > 1:
    sns.histplot(np.log1p(df['PRECTOTCORR'].clip(lower=0)), bins=30, kde=True)
    plt.title('Log-Scaled Distribution of PRECTOTCORR')
    plt.xlabel('log(1 + PRECTOTCORR)')
else:
    sns.histplot(df['PRECTOTCORR'], bins=30, kde=True)
    plt.title('Distribution of PRECTOTCORR')
    plt.xlabel('PRECTOTCORR')
plt.show()
plt.figure(figsize=(9, 6))
sizes = (df['PRECTOTCORR'].clip(lower=0) + 1) * 2
plt.scatter(df['T2M'], df['RH2M'], s=sizes, alpha=0.35)
plt.title('Bubble Chart: T2M vs RH2M (Bubble = PRECTOTCORR)')
plt.xlabel('T2M (C)')
plt.ylabel('RH2M (%)')
plt.show()


### Distribution observation
Comment on precipitation skewness and the temperature-humidity-precipitation relationship visible in the bubble chart.